In [ ]:

"""Manage student records, validate scores, and generate results reports.

This program reads student records from students.txt, validates the data,
calculates averages and grades, maintains log files, supports searching
and editing records, and generates a class results report.
"""

import re


def validate_student_name(student_name):
    """Validate a student's name.

    Args:
        student_name: The name to validate.

    Returns:
        None if the name is valid.

    Raises:
        ValueError: If the name is empty or contains invalid characters.
    """
    if not student_name:
        raise ValueError("Student name is missing")

    if not re.fullmatch(
        r"[A-Za-z]+(?:[ '-][A-Za-z]+)*",
        student_name
    ):
        raise ValueError(
            "Student name must contain letters only, "
            "with spaces, hyphens, or apostrophes allowed"
        )


def error_log(record, message):
    """Write a validation error to errors.log.

    Args:
        record: The student record that caused the error.
        message: A description of the validation error.

    Returns:
        None.
    """
    with open("errors.log", "a") as error_file:
        error_file.write(f"Record: {record}\n")
        error_file.write(f"Error: {message}\n")
        error_file.write("-" * 60 + "\n")


def valid_stu_records_log(valid_record, message):
    """Append a valid student record and its results to valid.log.

    Args:
        valid_record: A list containing the student's five fields.
        message: The student's scores, average, grade, and category.

    Returns:
        None.
    """
    with open("valid.log", "a") as valid_file:
        valid_file.write("\n")
        valid_file.write("=" * 60 + "\n")
        valid_file.write("             VALID STUDENT RECORD\n")
        valid_file.write("=" * 60 + "\n")
        valid_file.write(f"Student ID   : {valid_record[0]}\n")
        valid_file.write(f"Student Name : {valid_record[1]}\n")
        valid_file.write(message + "\n")
        valid_file.write("-" * 60 + "\n")


def invalid_stu_records(invalid_record, message):
    """Append an invalid student record and its reason to invalid.log.

    Args:
        invalid_record: The student record that failed validation.
        message: A description of why the record is invalid.

    Returns:
        None.
    """
    with open("invalid.log", "a") as invalid_file:
        invalid_file.write("\n")
        invalid_file.write("=" * 60 + "\n")
        invalid_file.write("            INVALID STUDENT RECORD\n")
        invalid_file.write("=" * 60 + "\n")
        invalid_file.write(f"Record : {invalid_record}\n")
        invalid_file.write(f"Reason : {message}\n")
        invalid_file.write("-" * 60 + "\n")


def display_menu():
    """Display the student record management menu."""
    print("\n" + "=" * 60)
    print("          STUDENT RECORD MANAGEMENT SYSTEM")
    print("=" * 60)
    print("1. Show/Print Error Logs")
    print("2. Show/Print Valid Records")
    print("3. Show/Print Invalid Records")
    print("4. Add New Student Record")
    print("5. Search Student Record (using Student ID)")
    print("6. Edit Existing Record")
    print("7. Generate Results Report")
    print("8. Show Best Student and Lowest Average Student")
    print("9. Exit")
    print("=" * 60)


def display_file(file_name):
    """Display the contents of a specified file.

    Args:
        file_name: The name of the file to display.

    Returns:
        None.
    """
    try:
        with open(file_name, "r") as file:
            content = file.read()

            print("\n" + "=" * 60)
            print(f"             {file_name.upper()}")
            print("=" * 60)

            if content.strip():
                print(content)
            else:
                print("The file is empty.")

    except FileNotFoundError:
        print(f"\n{file_name} does not exist yet.")

    except PermissionError:
        print(f"\nPermission denied when reading {file_name}.")


def calculate_grade(average):
    """Determine the grade and performance category for an average score.

    Args:
        average: The student's average score across three subjects.

    Returns:
        A tuple containing the letter grade and performance category.
    """
    if average >= 70:
        grade = "A"
        category = "Excellent"
    elif average >= 60:
        grade = "B"
        category = "Very Good"
    elif average >= 50:
        grade = "C"
        category = "Good"
    elif average >= 45:
        grade = "D"
        category = "Pass"
    else:
        grade = "F"
        category = "Fail"

    return grade, category


def process_student_records(write_logs=False, log_only_last=False):
    """Read, validate, and calculate results for student records.

    Each record must contain an ID, a valid name, and three integer
    scores. IDs must be unique, and scores must be between 0 and 100.

    Args:
        write_logs: Whether to append validation results to log files.
        log_only_last: Whether to log only the last record in students.txt.

    Returns:
        None. Updates the global record lists, student IDs, and results.

    Raises:
        FileNotFoundError: If students.txt cannot be found.
        PermissionError: If students.txt cannot be accessed.
    """
    global valid_stu_records
    global invalid_stu_record
    global stu_ids
    global student_results

    valid_stu_records = []
    invalid_stu_record = []
    stu_ids = set()
    student_results = []

    with open("students.txt", "r") as stu_file:
        rows_list = stu_file.readlines()

    for row_index, rows in enumerate(rows_list):
        stu_data = [
            field.strip() for field in rows.strip().split(",")
        ]

        # Decide whether this record should be written to the logs.
        should_log = write_logs and (
            not log_only_last or row_index == len(rows_list) - 1
        )

        errors = []

        # Validate the number of fields.
        if len(stu_data) != 5:
            message = "Student record must contain exactly 5 fields"

            if should_log:
                error_log(stu_data, message)
                invalid_stu_records(stu_data, message)

            invalid_stu_record.append(stu_data)
            continue

        # Check for a missing student ID.
        if stu_data[0] == "":
            errors.append("Student ID is missing")

        # Validate the student name.
        try:
            validate_student_name(stu_data[1])

        except ValueError as exc:
            errors.append(str(exc))

        # Convert subject scores to integers.
        try:
            python_score = int(stu_data[2])
            sql_score = int(stu_data[3])
            exel_score = int(stu_data[4])

        except ValueError:
            message = "Invalid score: fields must contain integers"

            if should_log:
                error_log(stu_data, message)
                invalid_stu_records(stu_data, message)

            invalid_stu_record.append(stu_data)
            continue

        # Validate the subject score ranges.
        if not 0 <= python_score <= 100:
            errors.append(
                "Python score must be between 0 and 100"
            )

        if not 0 <= sql_score <= 100:
            errors.append(
                "SQL score must be between 0 and 100"
            )

        if not 0 <= exel_score <= 100:
            errors.append(
                "Excel score must be between 0 and 100"
            )

        # Check for duplicate student IDs.
        if stu_data[0] in stu_ids:
            errors.append("Duplicate student ID")

        # Log and reject records that fail validation.
        if errors:
            if should_log:
                for message in errors:
                    error_log(stu_data, message)

                invalid_stu_records(
                    stu_data, "; ".join(errors)
                )

            invalid_stu_record.append(stu_data)
            continue

        # Store valid records and their IDs.
        valid_stu_records.append(stu_data)
        stu_ids.add(stu_data[0])

        # Calculate the average score.
        average = (
            python_score + sql_score + exel_score
        ) / 3

        # Assign the grade and category.
        grade, category = calculate_grade(average)

        # Store the calculated results.
        student_results.append({
            "id": stu_data[0],
            "name": stu_data[1],
            "python": python_score,
            "sql": sql_score,
            "excel": exel_score,
            "average": average,
            "grade": grade,
            "category": category
        })

        # Save valid records to valid.log when requested.
        if should_log:
            valid_stu_records_log(
                stu_data,
                f"Python Score   : {python_score}\n"
                f"SQL Score      : {sql_score}\n"
                f"Excel Score    : {exel_score}\n"
                f"Average Score  : {average:.2f}\n"
                f"Grade          : {grade}\n"
                f"Category       : {category}"
            )


def search_student_by_id(student_results):
    """Find and display a valid student record by student ID.

    Args:
        student_results: A list of dictionaries containing student results.

    Returns:
        None.
    """
    search_id = input(
        "\nEnter the student ID to search: "
    ).strip()

    for student in student_results:
        if student["id"] == search_id:
            print("\n" + "=" * 60)
            print("              STUDENT SEARCH RESULT")
            print("=" * 60)
            print(f"Student ID       : {student['id']}")
            print(f"Student Name     : {student['name']}")
            print("-" * 60)
            print(f"Python Score     : {student['python']}")
            print(f"SQL Score        : {student['sql']}")
            print(f"Excel Score      : {student['excel']}")
            print("-" * 60)
            print(f"Average Score    : {student['average']:.2f}")
            print(f"Grade            : {student['grade']}")
            print(f"Category         : {student['category']}")
            print("=" * 60)
            return

    print(f"\nNo valid student record found for ID: {search_id}")


def add_new_student_record():
    """Add, validate, and immediately process a new student record.

    Valid records are appended to valid.log. Invalid records are
    appended to errors.log and invalid.log. Only the new record is
    added to the logs during this operation.

    Returns:
        None.

    Raises:
        OSError: If students.txt cannot be written.
    """
    print("\nEnter the new student's details.")

    # Refresh existing records and remember their IDs.
    process_student_records()
    existing_ids = set(stu_ids)

    student_id = input("Student ID: ").strip()

    # Keep requesting a name until validation succeeds.
    while True:
        student_name = input("Student Name: ").strip()

        try:
            validate_student_name(student_name)

        except ValueError as exc:
            print(f"Invalid name: {exc}")
            print("Please enter the student's name again.")

        else:
            break

    python_score = input("Python Score: ").strip()
    sql_score = input("SQL Score: ").strip()
    excel_score = input("Excel Score: ").strip()

    new_record = [
        student_id,
        student_name,
        python_score,
        sql_score,
        excel_score
    ]

    try:
        with open("students.txt", "a") as stu_file:
            stu_file.write(",".join(new_record) + "\n")

    except OSError as exc:
        raise OSError(
            "Unable to save the new student record to students.txt."
        ) from exc

    # Immediately validate every record but log only the new record.
    process_student_records(
        write_logs=True,
        log_only_last=True
    )

    # Confirm that the newly added record is valid.
    if (
        new_record in valid_stu_records
        and student_id not in existing_ids
    ):
        print("\nStudent record added and validated successfully.")
        print("The record has been appended to valid.log.")

    else:
        print("\nStudent record saved but failed validation.")
        print("Check errors.log and invalid.log for details.")


def edit_existing_record():
    """Find a student by ID and update the student's record.

    Blank fields retain their existing values. The updated record is
    saved to students.txt and the records are reprocessed.

    Returns:
        None.
    """
    search_id = input(
        "\nEnter the Student ID to edit: "
    ).strip()

    try:
        with open("students.txt", "r") as stu_file:
            rows = stu_file.readlines()

    except FileNotFoundError:
        print("students.txt cannot be found.")
        return

    except PermissionError:
        print("Permission denied when reading students.txt.")
        return

    record_found = False
    updated_rows = []

    for row in rows:
        stu_data = [
            field.strip() for field in row.strip().split(",")
        ]

        if len(stu_data) == 5 and stu_data[0] == search_id:
            record_found = True

            print("\nCurrent record:")
            print(",".join(stu_data))
            print("\nEnter the updated details.")
            print("Leave a field blank to keep its current value.")

            student_name = input(
                f"Student Name [{stu_data[1]}]: "
            ).strip()

            python_score = input(
                f"Python Score [{stu_data[2]}]: "
            ).strip()

            sql_score = input(
                f"SQL Score [{stu_data[3]}]: "
            ).strip()

            excel_score = input(
                f"Excel Score [{stu_data[4]}]: "
            ).strip()

            if student_name == "":
                student_name = stu_data[1]

            if python_score == "":
                python_score = stu_data[2]

            if sql_score == "":
                sql_score = stu_data[3]

            if excel_score == "":
                excel_score = stu_data[4]

            updated_record = [
                search_id,
                student_name,
                python_score,
                sql_score,
                excel_score
            ]

            updated_rows.append(
                ",".join(updated_record) + "\n"
            )

        else:
            updated_rows.append(row)

    if record_found:
        try:
            with open("students.txt", "w") as stu_file:
                stu_file.writelines(updated_rows)

        except OSError as exc:
            print(f"Unable to update students.txt: {exc}")
            return

        process_student_records()
        print("\nStudent record updated successfully.")

    else:
        print(
            f"\nStudent ID {search_id} was not found "
            "in students.txt."
        )


def show_best_and_lowest_student():
    """Display the students with the highest and lowest average scores.

    Returns:
        None.
    """
    # Refresh records to include recent changes.
    process_student_records()

    if not student_results:
        print("\nNo valid student records are available.")
        return

    # Find the highest and lowest averages.
    highest_student = max(
        student_results,
        key=lambda student: student["average"]
    )

    lowest_student = min(
        student_results,
        key=lambda student: student["average"]
    )

    print("\n" + "=" * 60)
    print("          BEST AND LOWEST-AVERAGE STUDENTS")
    print("=" * 60)

    # Display the best student.
    print("\nBEST STUDENT (HIGHEST AVERAGE)")
    print("-" * 60)
    print(f"Student ID       : {highest_student['id']}")
    print(f"Student Name     : {highest_student['name']}")
    print(f"Python Score     : {highest_student['python']}")
    print(f"SQL Score        : {highest_student['sql']}")
    print(f"Excel Score      : {highest_student['excel']}")
    print(f"Highest Average  : {highest_student['average']:.2f}")
    print(f"Grade            : {highest_student['grade']}")
    print(f"Category         : {highest_student['category']}")

    # Display the lowest-average student.
    print("\nLOWEST-AVERAGE STUDENT")
    print("-" * 60)
    print(f"Student ID       : {lowest_student['id']}")
    print(f"Student Name     : {lowest_student['name']}")
    print(f"Python Score     : {lowest_student['python']}")
    print(f"SQL Score        : {lowest_student['sql']}")
    print(f"Excel Score      : {lowest_student['excel']}")
    print(f"Lowest Average   : {lowest_student['average']:.2f}")
    print(f"Grade            : {lowest_student['grade']}")
    print(f"Category         : {lowest_student['category']}")

    print("=" * 60)


def generate_results_report():
    """Generate student results and a class summary report.

    The report is overwritten each time this function runs.

    Returns:
        None.
    """
    # Refresh records before generating the report.
    process_student_records()

    with open("results_report.txt", "w") as report_file:
        report_file.write("=" * 60 + "\n")
        report_file.write("              STUDENT RESULTS REPORT\n")
        report_file.write("=" * 60 + "\n")

        # Write individual student results.
        for student in student_results:
            report_file.write("\n")
            report_file.write("-" * 60 + "\n")
            report_file.write("                 STUDENT RECORD\n")
            report_file.write("-" * 60 + "\n")
            report_file.write(
                f"Student ID       : {student['id']}\n"
            )
            report_file.write(
                f"Student Name     : {student['name']}\n"
            )
            report_file.write("-" * 60 + "\n")
            report_file.write(
                f"Python Score     : {student['python']}\n"
            )
            report_file.write(
                f"SQL Score        : {student['sql']}\n"
            )
            report_file.write(
                f"Excel Score      : {student['excel']}\n"
            )
            report_file.write("-" * 60 + "\n")
            report_file.write(
                f"Average Score    : {student['average']:.2f}\n"
            )
            report_file.write(
                f"Grade            : {student['grade']}\n"
            )
            report_file.write(
                f"Category         : {student['category']}\n"
            )
            report_file.write("=" * 60 + "\n")

        # Calculate the number of valid and invalid records.
        total_records = (
            len(valid_stu_records) + len(invalid_stu_record)
        )

        report_file.write("\nCLASS SUMMARY\n")
        report_file.write("=" * 60 + "\n")
        report_file.write(
            f"Total Records Read : {total_records}\n"
        )
        report_file.write(
            f"Valid Records      : {len(valid_stu_records)}\n"
        )
        report_file.write(
            f"Invalid Records    : {len(invalid_stu_record)}\n"
        )

        # Calculate class statistics when valid records exist.
        if student_results:
            class_average = sum(
                student["average"] for student in student_results
            ) / len(student_results)

            highest_student = max(
                student_results,
                key=lambda student: student["average"]
            )

            lowest_student = min(
                student_results,
                key=lambda student: student["average"]
            )

            report_file.write(
                f"Class Average      : {class_average:.2f}\n"
            )
            report_file.write(
                f"Highest Average    : {highest_student['average']:.2f}\n"
            )
            report_file.write(
                f"Top Student        : {highest_student['name']}\n"
            )
            report_file.write(
                f"Top Student ID     : {highest_student['id']}\n"
            )
            report_file.write(
                f"Lowest Average     : {lowest_student['average']:.2f}\n"
            )
            report_file.write(
                f"Lowest Student     : {lowest_student['name']}\n"
            )
            report_file.write(
                f"Lowest Student ID  : {lowest_student['id']}\n"
            )

        else:
            # Show N/A when there are no valid students.
            report_file.write("Class Average      : N/A\n")
            report_file.write("Highest Average    : N/A\n")
            report_file.write("Top Student        : N/A\n")
            report_file.write("Top Student ID     : N/A\n")
            report_file.write("Lowest Average     : N/A\n")
            report_file.write("Lowest Student     : N/A\n")
            report_file.write("Lowest Student ID  : N/A\n")

        report_file.write("=" * 60 + "\n")
        report_file.write("             END OF REPORT\n")
        report_file.write("=" * 60 + "\n")

    print("\nResults report generated successfully.")
    print("Report saved to results_report.txt.")


def main():
    """Run the student record management menu.

    Loads student records, displays the menu, and calls the appropriate
    function based on the user's selection.

    Returns:
        None.
    """
    try:
        # Create students.txt if it does not exist.
        with open("students.txt", "a"):
            pass

        # Load and validate student records at startup.
        process_student_records(write_logs=True)

        print("\nStudent records loaded successfully.")

        # Continue displaying the menu until the user exits.
        while True:
            display_menu()

            choice = input("Enter your choice (1-9): ").strip()

            if choice == "1":
                display_file("errors.log")

            elif choice == "2":
                display_file("valid.log")

            elif choice == "3":
                display_file("invalid.log")

            elif choice == "4":
                add_new_student_record()

            elif choice == "5":
                process_student_records()
                search_student_by_id(student_results)

            elif choice == "6":
                edit_existing_record()

            elif choice == "7":
                generate_results_report()

            elif choice == "8":
                show_best_and_lowest_student()

            elif choice == "9":
                print("\nExiting Student Record Management System.")
                break

            else:
                print(
                    "\nInvalid choice. Please enter a number from 1 to 9."
                )

    except FileNotFoundError:
        print("The required file cannot be found.")

    except PermissionError:
        print("Permission denied. Please check your file permissions.")

    except Exception as exc:
        print(f"An unexpected error occurred: {exc}")


if __name__ == "__main__":
    main()



Student records loaded successfully.

          STUDENT RECORD MANAGEMENT SYSTEM
1. Show/Print Error Logs
2. Show/Print Valid Records
3. Show/Print Invalid Records
4. Add New Student Record
5. Search Student Record (using Student ID)
6. Edit Existing Record
7. Generate Results Report
8. Show Best Student and Lowest Average Student
9. Exit


Enter your choice (1-9):  4



Enter the new student's details.


Student ID:  ST30
Student Name:  John bull
Python Score:  89
SQL Score:  81
Excel Score:  93



Student record added and validated successfully.
The record has been appended to valid.log.

          STUDENT RECORD MANAGEMENT SYSTEM
1. Show/Print Error Logs
2. Show/Print Valid Records
3. Show/Print Invalid Records
4. Add New Student Record
5. Search Student Record (using Student ID)
6. Edit Existing Record
7. Generate Results Report
8. Show Best Student and Lowest Average Student
9. Exit


Enter your choice (1-9):  2



             VALID.LOG

             VALID STUDENT RECORD
Student ID   : ST001
Student Name : Amaka Okafor
Python Score   : 78
SQL Score      : 84
Excel Score    : 91
Average Score  : 84.33
Grade          : A
Category       : Excellent
------------------------------------------------------------

             VALID STUDENT RECORD
Student ID   : ST002
Student Name : David Bello
Python Score   : 65
SQL Score      : 72
Excel Score    : 69
Average Score  : 68.67
Grade          : B
Category       : Very Good
------------------------------------------------------------

             VALID STUDENT RECORD
Student ID   : ST003
Student Name : John Ade
Python Score   : 90
SQL Score      : 88
Excel Score    : 94
Average Score  : 90.67
Grade          : A
Category       : Excellent
------------------------------------------------------------

             VALID STUDENT RECORD
Student ID   : ST004
Student Name : Chioma Eze
Python Score   : 82
SQL Score      : 79
Excel Score    : 87
Average Score  : 